In [7]:
import numpy as np
from sklearn.datasets import load_iris


iris = load_iris()
# X — ознаки (довжина/ширина чашолистка та пелюстки)
X = iris.data
# y — класи (0,1,2) -> тип квітки
y = iris.target
# назви ознак
feature_names = iris.feature_names
# назви класів
target_names = iris.target_names

print("Розмір X:", X.shape)
print("Розмір y:", y.shape)
print("Назви ознак:", feature_names)
print("Назви класів:", target_names)
print("Унікальні класи:", np.unique(y))

Розмір X: (150, 4)
Розмір y: (150,)
Назви ознак: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Назви класів: ['setosa' 'versicolor' 'virginica']
Унікальні класи: [0 1 2]


1. Поділ на навчальну і тестову вибірки

In [40]:
from sklearn.model_selection import train_test_split


# train — модель вчиться
# test — модель перевіряється
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3, # 30% даних для тесту
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (105, 4)
y_train: (105,)
X_test: (45, 4)
y_test: (45,)


2. Виділення ознак окремо для кожного класу

In [8]:
# отримуємо список класів [0,1,2]
classes = np.unique(y_train)

# створюємо словник:
# ключ = клас, значення = всі об'єкти цього класу
X_train_by_class = {}
for cls in classes:
    # беремо всі рядки X_train, де y_train == cls
    X_train_by_class[cls] = X_train[y_train == cls]

for cls in classes:
    print(f"Клас {cls} ({target_names[cls]}):", X_train_by_class[cls].shape)

Клас 0 (setosa): (35, 4)
Клас 1 (versicolor): (35, 4)
Клас 2 (virginica): (35, 4)


3. Будуємо модель:

3.1. Обчислення середніх векторів для кожного класу

In [16]:
# середній вектор = "центр" класу, наприклад: середня довжина пелюстки і т.д.
means = {}

for cls in classes:
    # axis=0 — беремо середнє по стовпцях (ознаках)
    means[cls] = np.mean(X_train_by_class[cls], axis=0)

for cls in classes:
    print(f"\nСередній вектор для класу {cls} ({target_names[cls]}):")
    print(means[cls])


Середній вектор для класу 0 (setosa):
[4.98857143 3.42571429 1.48571429 0.24      ]

Середній вектор для класу 1 (versicolor):
[5.94857143 2.73142857 4.23714286 1.30857143]

Середній вектор для класу 2 (virginica):
[6.68285714 3.00857143 5.63142857 2.06857143]


3.2. Обчислення матриць коваріації для кожного класу

In [9]:
# Коваріація показує як ознаки "рухаються разом".
# Наприклад: якщо одна росте — чи росте інша.
cov_matrices = {}

for cls in classes:
    # rowvar=False — означає:
    # рядки = об'єкти
    # стовпці = ознаки
    cov_matrices[cls] = np.cov(X_train_by_class[cls], rowvar=False)

for cls in classes:
    print(f"\nКоваріаційна матриця для класу {cls} ({target_names[cls]}):")
    print(cov_matrices[cls])


Коваріаційна матриця для класу 0 (setosa):
[[0.10633613 0.10794958 0.00894958 0.01282353]
 [0.10794958 0.17902521 0.01567227 0.01247059]
 [0.00894958 0.01567227 0.02361345 0.00382353]
 [0.01282353 0.01247059 0.00382353 0.00952941]]

Коваріаційна матриця для класу 1 (versicolor):
[[0.24786555 0.06989916 0.1722605  0.04810084]
 [0.06989916 0.08810084 0.06115126 0.03148739]
 [0.1722605  0.06115126 0.21769748 0.06584874]
 [0.04810084 0.03148739 0.06584874 0.03668908]]

Коваріаційна матриця для класу 2 (virginica):
[[0.43734454 0.10632773 0.33584874 0.0347395 ]
 [0.10632773 0.12080672 0.09148739 0.04380672]
 [0.33584874 0.09148739 0.33221849 0.0492521 ]
 [0.0347395  0.04380672 0.0492521  0.0657479 ]]


3.3. Обчислення обернених матриць коваріації та визначників

In [10]:
inv_cov_matrices = {}
det_cov_matrices = {}

for cls in classes:
    cov = cov_matrices[cls]

    # Невелика регуляризація на випадок числових проблем
    # додаємо маленьке число до діагоналі щоб матриця точно була оберненою (числова стабільність)
    cov_reg = cov + 1e-6 * np.eye(cov.shape[0])
    # обернена матриця — потрібна для "відстані Махаланобіса"
    inv_cov_matrices[cls] = np.linalg.inv(cov_reg)
    # визначник — показує "об'єм розсіювання"
    det_cov_matrices[cls] = np.linalg.det(cov_reg)

for cls in classes:
    print(f"\nКлас {cls} ({target_names[cls]}):")
    print("Обернена матриця коваріації:")
    print(inv_cov_matrices[cls])
    print("Визначник:", det_cov_matrices[cls])


Клас 0 (setosa):
Обернена матриця коваріації:
[[ 26.47135648 -15.00906171   2.69078838 -17.05830792]
 [-15.00906171  14.85709226  -4.59252418   2.59717168]
 [  2.69078838  -4.59252418  47.04429183 -16.48503372]
 [-17.05830792   2.59717168 -16.48503372 131.09511917]]
Визначник: 1.2524801036611766e-06

Клас 1 (versicolor):
Обернена матриця коваріації:
[[  9.9352957   -4.01775473  -8.39169197   5.48363263]
 [ -4.01775473  18.05507815   2.6274855  -14.94318551]
 [ -8.39169197   2.6274855   17.17207685 -22.0725861 ]
 [  5.48363263 -14.94318551 -22.0725861   72.50480061]]
Визначник: 2.2367758979503352e-05

Клас 2 (virginica):
Обернена матриця коваріації:
[[ 11.35047971  -3.07010281 -11.29785654   4.51147279]
 [ -3.07010281  13.33601344   0.57138821  -7.69131187]
 [-11.29785654   0.57138821  15.12516728  -5.741466  ]
 [  4.51147279  -7.69131187  -5.741466    22.25108188]]
Визначник: 0.0001367948733383223


3.4. Обчислення апріорних імовірностей класів

In [11]:
# P(клас) = частка об'єктів цього класу в train
priors = {}

n_train = len(y_train)

for cls in classes:
    priors[cls] = np.sum(y_train == cls) / n_train

print("Апріорні ймовірності класів:")
for cls in classes:
    print(f"Клас {cls} ({target_names[cls]}): {priors[cls]:.4f}")

Апріорні ймовірності класів:
Клас 0 (setosa): 0.3333
Клас 1 (versicolor): 0.3333
Клас 2 (virginica): 0.3333


4. Функція дискримінантної функції для одного рядка і одного класу

In [12]:
def qda_discriminant_one_class(x, mean, inv_cov, det_cov, prior):
    """
    QDA:
    1. наскільки x близький до центру класу (mean), наскільки x схожий на клас k?
    2. враховує форму розподілу (covariance)
    3. враховує популярність класу (prior)
    """
    diff = x - mean

    term1 = -0.5 * diff.T @ inv_cov @ diff  # відстань
    term2 = -0.5 * np.log(det_cov) # штраф за розкиданість
    term3 = np.log(prior) # апріорна ймовірність

    return term1 + term2 + term3

5. Функція для одного тестового об’єкта: значення всіх дискримінантних функцій

In [13]:
def qda_predict_one(x, means, inv_cov_matrices, det_cov_matrices, priors):
    # Для кожного класу рахуємо g_k(x) і беремо максимум
    scores = []

    for cls in sorted(means.keys()):
        score = qda_discriminant_one_class(
            x=x,
            mean=means[cls],
            inv_cov=inv_cov_matrices[cls],
            det_cov=det_cov_matrices[cls],
            prior=priors[cls]
        )
        scores.append(score)

    scores = np.array(scores)
    # клас з найбільшим значенням
    class_labels = sorted(means.keys())
    predicted_class = class_labels[np.argmax(scores)]

    return predicted_class, scores

6. Перевірка на одному прикладі

In [17]:
sample_x = X_test[0]
sample_true = y_test[0]

pred_class, scores = qda_predict_one(
    sample_x,
    means,
    inv_cov_matrices,
    det_cov_matrices,
    priors
)

print("Тестовий приклад:", sample_x)
print("Справжній клас:", sample_true, "-", target_names[sample_true])
print("Прогнозований клас:", pred_class, "-", target_names[pred_class])
print("Значення дискримінантних функцій:", scores)

Тестовий приклад: [7.3 2.9 6.3 1.8]
Справжній клас: 2 - virginica
Прогнозований клас: 2 - virginica
Значення дискримінантних функцій: [-644.12606367   -6.99770519    1.36533489]


7. Функція softmax для перетворення значень у “ймовірності”

In [14]:
def softmax(scores):
    """
    Перетворює "оцінки" у ймовірності
    Наприклад: [2.3, 1.2, 0.1] -> [0.7, 0.2, 0.1]
    """
    # віднімаємо максимум — щоб уникнути overflow
    scores_shifted = scores - np.max(scores)  # для числової стабільності
    exp_scores = np.exp(scores_shifted)
    return exp_scores / np.sum(exp_scores)

8. Функція для всієї тестової вибірки

In [18]:
def qda_predict_all(X, means, inv_cov_matrices, det_cov_matrices, priors):
    predictions = []
    probabilities = []
    all_scores = []

    for x in X:
        pred_class, scores = qda_predict_one(
            x,
            means,
            inv_cov_matrices,
            det_cov_matrices,
            priors
        )

        probs = softmax(scores)

        predictions.append(pred_class)
        probabilities.append(probs)
        all_scores.append(scores)

    return np.array(predictions), np.array(probabilities), np.array(all_scores)

9. Прогнозування власною реалізацією

In [19]:
y_pred_manual, y_proba_manual, manual_scores = qda_predict_all(
    X_test,
    means,
    inv_cov_matrices,
    det_cov_matrices,
    priors
)

print("Перші 10 прогнозів власної реалізації:")
print(y_pred_manual[:10])

print("\nПерші 5 рядків імовірностей:")
print(np.round(y_proba_manual[:5], 4))

Перші 10 прогнозів власної реалізації:
[2 1 1 1 2 2 1 1 0 2]

Перші 5 рядків імовірностей:
[[0.000e+00 2.000e-04 9.998e-01]
 [0.000e+00 9.923e-01 7.700e-03]
 [0.000e+00 7.679e-01 2.321e-01]
 [0.000e+00 9.913e-01 8.700e-03]
 [0.000e+00 1.700e-01 8.300e-01]]


10. Прогнозування через sklearn

In [21]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis


# Створюємо об'єкт моделі QDA
qda_sklearn = QuadraticDiscriminantAnalysis()

# Тут відбувається "навчання", sklearn сам обчислює:
# - середні вектори μ_k для кожного класу
# - матриці коваріації Σ_k
# - апріорні ймовірності P(ω_k)
# - внутрішні параметри для дискримінантної функції
qda_sklearn.fit(X_train, y_train)

# Для кожного об'єкта з X_test:
# 1. модель рахує значення дискримінантної функції g_k(x) для кожного класу
# 2. вибирає клас з найбільшим значенням
# 3. повертає номер класу (0, 1 або 2)
y_pred_sklearn = qda_sklearn.predict(X_test)
# Для кожного об'єкта модель повертає:  [P(Клас 0 = setosa), P(Клас 1 = versicolor), P(Клас 2 = virginica)]
# тобто це розподіл ймовірностей, який показує: наскільки кожен клас “підходить” цьому об’єкту
y_proba_sklearn = qda_sklearn.predict_proba(X_test)

print("Перші 10 прогнозів sklearn:")
print(y_pred_sklearn[:10])

print("\nПерші 5 рядків імовірностей sklearn:")
print(np.round(y_proba_sklearn[:5], 4))

Перші 10 прогнозів sklearn:
[2 1 1 1 2 2 1 1 0 2]

Перші 5 рядків імовірностей sklearn:
[[0.000e+00 2.000e-04 9.998e-01]
 [0.000e+00 9.923e-01 7.700e-03]
 [0.000e+00 7.679e-01 2.321e-01]
 [0.000e+00 9.913e-01 8.700e-03]
 [0.000e+00 1.700e-01 8.300e-01]]


11. Порівняння результатів

In [23]:
from sklearn.metrics import accuracy_score


accuracy_manual = accuracy_score(y_test, y_pred_manual)
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)

print("Accuracy власної реалізації:", round(accuracy_manual, 4))
print("Accuracy sklearn QDA:", round(accuracy_sklearn, 4))

Accuracy власної реалізації: 0.9778
Accuracy sklearn QDA: 0.9778


12. Порівняння прогнозів пооб’єктно

In [25]:
import pandas as pd


comparison_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred_manual": y_pred_manual,
    "y_pred_sklearn": y_pred_sklearn,
    "manual_name": [target_names[i] for i in y_pred_manual],
    "sklearn_name": [target_names[i] for i in y_pred_sklearn],
    "true_name": [target_names[i] for i in y_test]
})

comparison_df["manual_equals_sklearn"] = comparison_df["y_pred_manual"] == comparison_df["y_pred_sklearn"]
comparison_df["manual_equals_true"] = comparison_df["y_pred_manual"] == comparison_df["y_true"]
comparison_df["sklearn_equals_true"] = comparison_df["y_pred_sklearn"] == comparison_df["y_true"]

comparison_df.head(15)

,y_true,y_pred_manual,y_pred_sklearn,manual_name,sklearn_name,true_name,manual_equals_sklearn,manual_equals_true,sklearn_equals_true
0,2,2,2,virginica,virginica,virginica,True,True,True
1,1,1,1,versicolor,versicolor,versicolor,True,True,True
2,2,1,1,versicolor,versicolor,virginica,True,False,False
3,1,1,1,versicolor,versicolor,versicolor,True,True,True
4,2,2,2,virginica,virginica,virginica,True,True,True
5,2,2,2,virginica,virginica,virginica,True,True,True
6,1,1,1,versicolor,versicolor,versicolor,True,True,True
7,1,1,1,versicolor,versicolor,versicolor,True,True,True
8,0,0,0,setosa,setosa,setosa,True,True,True
9,2,2,2,virginica,virginica,virginica,True,True,True


13. Скільки прогнозів співпало

In [26]:
# скільки однакових прогнозів
same_predictions = np.sum(y_pred_manual == y_pred_sklearn)
total_predictions = len(y_test)

print(f"Кількість однакових прогнозів: {same_predictions} з {total_predictions}")
print(f"Частка однакових прогнозів: {same_predictions / total_predictions:.4f}")

Кількість однакових прогнозів: 45 з 45
Частка однакових прогнозів: 1.0000


14. Classification report

In [28]:
from sklearn.metrics import classification_report


print("=== Власна реалізація ===")
print(classification_report(y_test, y_pred_manual, target_names=target_names))

print("=== sklearn QDA ===")
print(classification_report(y_test, y_pred_sklearn, target_names=target_names))

=== Власна реалізація ===
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.94      1.00      0.97        15
   virginica       1.00      0.93      0.97        15

    accuracy                           0.98        45
   macro avg       0.98      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45

=== sklearn QDA ===
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.94      1.00      0.97        15
   virginica       1.00      0.93      0.97        15

    accuracy                           0.98        45
   macro avg       0.98      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45



15. Матриці помилок

In [30]:
from sklearn.metrics import confusion_matrix


print("Матриця помилок - власна реалізація:")
print(confusion_matrix(y_test, y_pred_manual))

print("\nМатриця помилок - sklearn:")
print(confusion_matrix(y_test, y_pred_sklearn))

Матриця помилок - власна реалізація:
[[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]

Матриця помилок - sklearn:
[[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]


16. Порівняння ймовірностей

In [36]:
proba_comparison = pd.DataFrame({
    "manual_p_setosa": y_proba_manual[:, 0],
    "manual_p_versicolor": y_proba_manual[:, 1],
    "manual_p_virginica": y_proba_manual[:, 2],
    "sklearn_p_setosa": y_proba_sklearn[:, 0],
    "sklearn_p_versicolor": y_proba_sklearn[:, 1],
    "sklearn_p_virginica": y_proba_sklearn[:, 2],
})

print('\nПорівння оцінки ймовірностей приналежності об’єктів до кожного класу:')
proba_comparison.head(10)


Порівння оцінки ймовірностей приналежності об’єктів до кожного класу:


,manual_p_setosa,manual_p_versicolor,manual_p_virginica,sklearn_p_setosa,sklearn_p_versicolor,sklearn_p_virginica
0,4.640301e-281,2.332795e-04,9.997667e-01,4.508824e-281,2.332659e-04,9.997667e-01
1,5.879362e-126,9.923218e-01,7.678230e-03,5.800700e-126,9.923224e-01,7.677627e-03
2,1.282246e-158,7.679176e-01,2.320824e-01,1.260934e-158,7.679236e-01,2.320764e-01
3,1.948697e-129,9.913165e-01,8.683509e-03,1.920073e-129,9.913174e-01,8.682611e-03
4,7.147493e-154,1.699925e-01,8.300075e-01,7.015470e-154,1.699762e-01,8.300238e-01
5,7.363124e-245,1.035178e-08,1.000000e+00,7.132834e-245,1.034000e-08,1.000000e+00
6,5.531566e-88,9.979910e-01,2.009031e-03,5.476126e-88,9.979910e-01,2.008967e-03
7,1.538256e-75,9.999770e-01,2.304348e-05,1.525961e-75,9.999770e-01,2.304027e-05
8,1.000000e+00,3.543292e-18,2.188247e-36,1.000000e+00,3.540308e-18,2.185689e-36
9,2.336190e-204,2.577825e-07,9.999997e-01,2.273773e-204,2.574297e-07,9.999997e-01


In [37]:
values_qda = qda_sklearn._decision_function(X_test)

decision_df = pd.DataFrame({
    "g0_manual": manual_scores[:, 0],
    "g1_manual": manual_scores[:, 1],
    "g2_manual": manual_scores[:, 2],
    "g0_sklearn": values_qda[:, 0],
    "g1_sklearn": values_qda[:, 1],
    "g2_sklearn": values_qda[:, 2],
})

print("\nПорівняння значень дискримінантної функції:")
decision_df.head(10)


Порівняння значень дискримінантної функції:


,g0_manual,g1_manual,g2_manual,g0_sklearn,g1_sklearn,g2_sklearn
0,-644.126064,-6.997705,1.365335,-644.154807,-6.997763,1.365335
1,-285.029972,3.316593,-1.545065,-285.043417,3.316619,-1.545118
2,-361.458259,1.837500,0.640910,-361.475013,1.837514,0.640890
3,-294.184384,2.173210,-2.564398,-294.199151,2.173243,-2.564470
4,-350.604415,0.254926,1.840607,-350.623067,0.254822,1.840618
5,-559.925931,-16.175175,2.210933,-559.957705,-16.176312,2.210934
6,-198.344626,2.570381,-3.637711,-198.354699,2.570380,-3.637743
7,-168.667937,3.595273,-7.082832,-168.675923,3.595311,-7.082932
8,1.420218,-38.761258,-80.689745,1.420022,-38.762296,-80.691110
9,-470.569057,-16.861369,-1.690219,-470.596256,-16.862856,-1.690337


**Висновок**

У ході виконання роботи було реалізовано власний алгоритм Quadratic Discriminant Analysis (QDA) для класифікації об'єктів датасету Iris. Для кожного класу окремо було обчислено:
- середній вектор ознак;
- матрицю коваріації;
- обернену матрицю коваріації;
- визначник матриці коваріації;
- апріорну ймовірність класу.

На основі цих параметрів було реалізовано дискримінантну функцію QDA для одного об'єкта та для всієї тестової вибірки. Також було отримано оцінки ймовірностей належності до класів шляхом нормалізації значень дискримінантної функції через softmax.

Після цього результати власної реалізації були порівняні з результатами моделі QuadraticDiscriminantAnalysis з бібліотеки sklearn яка обчислює апостеріорні ймовірності.

  Точність власної реалізації: 0.9778

  Точність sklearn QDA: 0.9778

  Кількість однакових прогнозів: 45 з 45

  Частка однакових прогнозів: 1

Отримані результати показують, що власна реалізація дає дуже близькі або однакові результати порівняно з бібліотечною реалізацією sklearn. Можливі незначні відмінності можуть пояснюватися числовою точністю обчислень, способом регуляризації матриць коваріації та методом обчислення ймовірностей приналежності до класів.